In [32]:
from sklearn.datasets import fetch_openml

housing = fetch_openml(name= "house_prices", as_frame= True)
df = housing.frame
df.to_csv("../data/raw/ames_housing.csv", index = False)

print(f"shape: {df.shape}")
print(f"columns: {df.columns}")

shape: (1460, 81)
columns: Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
     

In [33]:
import sys
sys.path.append("..")
from src.data_utils import load_data
from src.cleaning import fill_absent

In [34]:
df = load_data("../data/raw/ames_housing.csv")
print(f"shape: {df.shape}")
missing = df.isnull().sum()
print(missing[missing>0].sort_values(ascending=False))

shape: (1460, 81)
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [35]:
blank_rows = df[df["PoolQC"].isnull()]
print(blank_rows["PoolArea"].value_counts())

PoolArea
0    1453
Name: count, dtype: int64


In [36]:
blank_rows = df[df["GarageType"].isnull()]
print(blank_rows["GarageArea"].value_counts())

GarageArea
0    81
Name: count, dtype: int64


In [37]:
garage_columns =["GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond"]
print(f"any: {df[garage_columns].isnull().any(axis=1).sum()}")
print(f"all: {df[garage_columns].isnull().all(axis=1).sum()}")

any: 81
all: 81


In [38]:
bst_col = ["BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtExposure"]
print(f"any: {df[bst_col].isnull().any(axis=1).sum()}")
print(f"all: {df[bst_col].isnull().all(axis=1).sum()}")

any: 39
all: 37


In [39]:
any_blank = df[bst_col].isnull().any(axis=1)
all_blank = df[bst_col].isnull().all(axis=1)

odd_ones = df[ any_blank & ~all_blank]
print(odd_ones[["BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtExposure", "TotalBsmtSF"]])

    BsmtFinType1 BsmtFinType2 BsmtQual BsmtCond BsmtExposure  TotalBsmtSF
332          GLQ          NaN       Gd       TA           No         3206
948          Unf          Unf       Gd       TA          NaN          936


In [40]:
blank_rows = df[df["LotFrontage"].isnull()]
print(blank_rows["LotArea"].describe())

count       259.000000
mean      13137.370656
std       16215.264451
min        1974.000000
25%        8065.500000
50%       10624.000000
75%       13018.500000
max      164660.000000
Name: LotArea, dtype: float64


In [41]:
blank_rows = df[df["MasVnrType"].isnull()]
print(blank_rows["MasVnrArea"].value_counts(dropna=False))


MasVnrArea
0.0      859
NaN        8
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64


|Columns|Blanks|What a blank means|Evidence|Treatment|
|--------|-----|---------------|----------|-------|
|PoolQC|1453|Pool is absent|PoolArea=0|Fill with the label None|
|GarageType|81|Garage is absent|GarageArea=0|Fill with the label None|
|GarageYrBlt|81|Garage is absent|GarageArea=0. Has a correlation of 0.83 with year built and 79% of values match in the two columns|leave out the whole column for our first model. You don't invent data that doesn't exist but you lose a feature that may add on to your model|
|GarageFinish|81|Garage is absent|GarageArea=0|Fill with the label None|
|GarageQual|81|Garage is absent|GarageArea=0|Fill with the label None|
|GarageCond|81|Garage is absent|GarageArea=0|Fill with the label None|
|BsmtExposure|37|Bsmt is absent|TotalBsmtSF=0|Fill with the label None|
|BsmtExposure|1|unknown|TotalBsmtSF>0|Fill with mode from training set|
|BsmtFinType2|37|Bsmt is absent|TotalBsmtSF=0|Fill with the label None|
|BsmtFinType2|1|unknown|TotalBsmtSF>0|Fill with mode from training set|
|BsmtQual|37|bsmt is absent|TotalBsmtSF=0|Fill with the label None|
|BsmtCond|37|bsmt is absent|TotalBsmtSF=0|Fill with the label None|
|BsmtFinType1|37|Bsmt is absent|all related bsmt columns also blank|Fill with the label None|
|LotFrontage|259|Unknown|LotArea>0|Fill with median of training set|
|MasVnrType|859|MasVnr is absent|MasVnrArea=0|Fill with the label None|
|MasVnrType|8|cannot tell|both MasVnr Area and type columns missing|Fill with training set mode?|
|MasVnrType|3|Unknown|MasVnrArea>0|Fill with the mode of the training set|
|MasVnrType|2|Ambiguous|MasVnrArea=1|fill with mode of training set?|
|Alley|1369|Absent|Dictionary (NA means No Alley Access)|Fill with the label None|
|MiscFeature|1406|Absent|MiscVal=0|Fill with the label None|
|FireplaceQu|690|Absent|Fireplaces=0|Fill with the label None|
|Fence|1179|Absent|Dictionary (NA means no fence)|Fill with the label None|
|Electrical|1|Unknown|Dictionary has no NA option|Fill with the training set mode|
|MasVnrArea|8|cannot tell|both MasVnr Area and type columns missing?|Fill with median from training set|


In [42]:
blank_rows = df[df["BsmtQual"].isnull()]
print(blank_rows["TotalBsmtSF"].value_counts())

TotalBsmtSF
0    37
Name: count, dtype: int64


In [43]:
clean_df = fill_absent(df, ['PoolQC'])
print(f"original: {df['PoolQC'].isnull().sum()}")
print(f"new: {clean_df['PoolQC'].isnull().sum()}")


original: 1453
new: 0


In [44]:
clean_df = fill_absent(df, ["BsmtExposure"]) 
print(f"original: {df["BsmtExposure"].isnull().sum()}")
print(f" new: {clean_df["BsmtExposure"].isnull().sum()}")

original: 38
 new: 0


In [45]:
df_work = df.copy()
mask = (df_work["BsmtExposure"].isnull() ) & (df_work["TotalBsmtSF"] == 0)
df_work.loc[mask, "BsmtExposure"] = "None"
print(f"mask sum: { mask.sum()}")
print(df_work["BsmtExposure"].isnull().sum())

mask sum: 37
1


In [46]:
bsmt_df = fill_absent(df,["BsmtExposure"], only_if_zero="TotalBsmtSF")
print(f" Bsmt blanks: {bsmt_df["BsmtExposure"].isnull().sum()}")
pool_df = fill_absent(df, ["PoolQC"])
print(f"pool blanks: {pool_df["PoolQC"].isnull().sum()}")

 Bsmt blanks: 1
pool blanks: 0


In [47]:
blank_rows = df[df["FireplaceQu"].isnull()]
print(blank_rows["Fireplaces"].value_counts())

Fireplaces
0    690
Name: count, dtype: int64


In [48]:
blank_rows = df[df["MiscFeature"].isnull()]
print(blank_rows["MiscVal"].value_counts())

MiscVal
0    1406
Name: count, dtype: int64


In [49]:
description = housing.DESCR
pos = description.find("Alley")
print(description[pos:pos+300])

Alley: Type of alley access to property

       Grvl	Gravel
       Pave	Paved
       NA 	No alley access
		
LotShape: General shape of property

       Reg	Regular	
       IR1	Slightly irregular
       IR2	Moderately Irregular
       IR3	Irregular
       
LandContour: Flatness of the property

     


In [50]:
pos = description.find("Fence")
print(description[pos:pos+300])

Fence: Fence quality
		
       GdPrv	Good Privacy
       MnPrv	Minimum Privacy
       GdWo	Good Wood
       MnWw	Minimum Wood/Wire
       NA	No Fence
	
MiscFeature: Miscellaneous feature not covered in other categories
		
       Elev	Elevator
       Gar2	2nd Garage (if not described in garage sectio


In [51]:
pos = description.find("Electrical:")
print(description[pos:pos+300])

Electrical: Electrical system

       SBrkr	Standard Circuit Breakers & Romex
       FuseA	Fuse Box over 60 AMP and all Romex wiring (Average)	
       FuseF	60 AMP Fuse Box and mostly Romex wiring (Fair)
       FuseP	60 AMP Fuse Box and mostly knob & tube wiring (poor)
       Mix	Mixed
		
1stFlrSF: 


In [52]:
matching_col = [col for col in df.columns if col.lower().startswith("garage")]
print(matching_col)

['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond']


In [59]:
filled = df[df["GarageYrBlt"].notna()]
print(filled["GarageYrBlt"].corr(filled["YearBuilt"]))
print((filled["GarageYrBlt"] == filled["YearBuilt"]).sum())

0.825667484174342
1089
